# Hackathon ONE — Team 16 — KnowledgeHub

Objetivo: classificar conteudos tecnicos em area principal e
subarea (dois níveis), extrair palavras-chave e recomendar conteudos relacionados.

Roteiro:
1. Carregamento da base
2. Analise exploratoria (EDA)
3. Tratamento de texto
4. Treinamento (dois niveis)
5. Avaliacao e metricas
6. Palavras-chave e recomendacao
7. Serializacao do modelo

## 1. Carregamento da base

Base produzida pela equipe. Cada conteudo tem uma area (nivel 1) e uma subarea (nivel 2). Os textos misturam dois estilos: documentacao tecnica e linguagem explicativa.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 100)
df = pd.read_csv("../dados/dataset.csv")
print(f"Registros: {len(df)}")
print(f"Areas: {df['area'].nunique()} | Subareas: {df['subarea'].nunique()}")
df.head()

## 2. Analise exploratoria (EDA)

In [ ]:
df.info()
print("\nNulos:\n", df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())

In [ ]:
# Distribuicao por area (nivel 1)
plt.figure(figsize=(8, 4))
df["area"].value_counts().plot(kind="barh", color="steelblue")
plt.title("Conteudos por area principal")
plt.xlabel("Quantidade")
plt.tight_layout(); plt.show()

In [ ]:
for area in sorted(df["area"].unique()):
    subs = df[df["area"] == area]["subarea"].value_counts()
    print(f"\n{area}:")
    for sub, n in subs.items():
        print(f"   {sub}: {n}")

In [ ]:
df["n_palavras"] = df["texto"].str.split().str.len()
print(df["n_palavras"].describe())
plt.figure(figsize=(8, 4))
plt.hist(df["n_palavras"], bins=25, color="steelblue", edgecolor="white")
plt.title("Distribuicao do tamanho dos textos")
plt.xlabel("Palavras"); plt.ylabel("Frequencia")
plt.tight_layout(); plt.show()

## 3. Tratamento de texto

Titulo e texto sao concatenados, depois,
TF-IDF transforma o texto em numeros: cada palavra vira uma coluna, com peso
alto quando ela aparece muito neste documento e pouco nos demais.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

STOPWORDS_PT = ["a","ao","aos","as","com","como","da","das","de","do","dos","e",
    "em","essa","esse","esta","este","eu","isso","na","nas","no","nos","o","os",
    "ou","para","pela","pelo","por","que","se","sem","ser","sobre","sua","suas",
    "seu","seus","tem","um","uma","neste","sao","conteudo","material","texto",
    "voce","ja","imagine","seguinte","trata","muita","gente","vamos","falar"]

X = df["titulo"].fillna("") + " " + df["texto"].fillna("")

demo = TfidfVectorizer(stop_words=STOPWORDS_PT, min_df=2)
m = demo.fit_transform(X)
print(f"Matriz: {m.shape}  (documentos x termos)")

## 4. Treinamento — dois niveis

Dois classificadores independentes, ambos TF-IDF + Regressao Logistica: um preve a area, outro a subarea. A Regressao Logistica foi escolhida por ser rapida, funcionar bem com TF-IDF e fornecer probabilidade (`predict_proba`).

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def novo_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=STOPWORDS_PT, ngram_range=(1,2),
                                  min_df=2, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, C=5.0)),
    ])

# Nivel 1: AREA
Xtr, Xte, ytr, yte = train_test_split(X, df["area"], test_size=0.2,
                                      random_state=42, stratify=df["area"])
pipe_area = novo_pipeline().fit(Xtr, ytr)
print("Area treinada.")

# Nivel 2: SUBAREA
Xtr2, Xte2, ytr2, yte2 = train_test_split(X, df["subarea"], test_size=0.2,
                                          random_state=42, stratify=df["subarea"])
pipe_sub = novo_pipeline().fit(Xtr2, ytr2)
print("Subarea treinada.")

## 5. Avaliacao e metricas

F1-score macro (trata todas as classes com o mesmo peso).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("=== AREA (nivel 1) ===")
print(classification_report(yte, pipe_area.predict(Xte), zero_division=0))

print("=== SUBAREA (nivel 2) ===")
print(classification_report(yte2, pipe_sub.predict(Xte2), zero_division=0))

In [ ]:
# Matriz AREA
rot = sorted(df["area"].unique())
mc = confusion_matrix(yte, pipe_area.predict(Xte), labels=rot)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(mc, display_labels=rot).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Matriz de confusao — Area principal")
plt.tight_layout(); plt.show()

In [ ]:
# Validacao cruzada nos dois niveis
sc_area = cross_val_score(novo_pipeline(), X, df["area"], cv=5, scoring="f1_macro")
sc_sub  = cross_val_score(novo_pipeline(), X, df["subarea"], cv=5, scoring="f1_macro")
print(f"F1-macro AREA:    {sc_area.mean():.3f} (+/- {sc_area.std():.3f})")
print(f"F1-macro SUBAREA: {sc_sub.mean():.3f} (+/- {sc_sub.std():.3f})")

In [ ]:
# Explicabilidade: termos que mais definem cada AREA
vet = pipe_area.named_steps["tfidf"]
clf = pipe_area.named_steps["clf"]
nomes = np.array(vet.get_feature_names_out())
pipe_area_full = novo_pipeline().fit(X, df["area"])
vet = pipe_area_full.named_steps["tfidf"]; clf = pipe_area_full.named_steps["clf"]
nomes = np.array(vet.get_feature_names_out())
for i, area in enumerate(clf.classes_):
    top = clf.coef_[i].argsort()[::-1][:6]
    print(f"{area}: {', '.join(nomes[top])}")

## 6. Palavras-chave e recomendacao

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# retreina para o modelo final
pipe_area = novo_pipeline().fit(X, df["area"])
pipe_sub  = novo_pipeline().fit(X, df["subarea"])
vetorizador = pipe_area.named_steps["tfidf"]
matriz_base = vetorizador.transform(X)

def palavras_chave(texto, n=5):
    v = vetorizador.transform([texto]); termos = np.array(vetorizador.get_feature_names_out())
    pesos = v.toarray()[0]; top = pesos.argsort()[::-1][:n]
    return [termos[i] for i in top if pesos[i] > 0]

def relacionados(texto, n=3):
    v = vetorizador.transform([texto]); sims = cosine_similarity(v, matriz_base)[0]
    out, vistos = [], set()
    for i in sims.argsort()[::-1]:
        if len(out) >= n or sims[i] <= 0.05: break
        if df["titulo"].iloc[i] in vistos: continue
        vistos.add(df["titulo"].iloc[i])
        out.append((df["titulo"].iloc[i], round(float(sims[i]), 3)))
    return out

# Exemplo 
exemplo = ("O que e machine learning e como ele funciona? Os robos possuem "
           "softwares que possibilitam o aprendizado a partir das interacoes.")
print("Area:", pipe_area.predict([exemplo])[0])
print("Subarea:", pipe_sub.predict([exemplo])[0])
print("Palavras-chave:", palavras_chave(exemplo))
print("Relacionados:")
for t, s in relacionados(exemplo):
    print(f"  {s} — {t}")

## 7. Serializacao do modelo

Salvamos os dois pipelines + a matriz da base num unico arquivo, pronto para a API e para o OCI Object Storage.

In [ ]:
import joblib
artefato = {
    "pipeline_area": pipe_area,
    "pipeline_subarea": pipe_sub,
    "vetorizador": vetorizador,
    "matriz_base": matriz_base,
    "titulos": df["titulo"].tolist(),
    "areas": df["area"].tolist(),
    "subareas": df["subarea"].tolist(),
}
joblib.dump(artefato, "../modelo/modelo_knowledgehub.joblib")
print("Modelo salvo.")

rec = joblib.load("../modelo/modelo_knowledgehub.joblib")
print("Teste apos recarregar — area:", rec["pipeline_area"].predict([exemplo])[0])

## Conclusao

- Base propria com 290 conteudos, 7 areas e 21 subareas, misturando textos
  tecnicos e explicativos.
- Classificacao em dois niveis: F1-macro alto na area, um pouco menor na subarea
  (esperado, por haver mais classes).
- Alem de classificar, a solucao extrai palavras-chave e recomenda conteudos
  relacionados por similaridade de cosseno.
- Modelo serializado com joblib, pronto para a API REST e o OCI Object Storage.

**saida da API**:
`area_principal`, `subarea`, `confianca_area`, `confianca_subarea`,
`palavras_chave`, `conteudos_relacionados`.